<a href="https://colab.research.google.com/github/ygarces-ctrl/Biomass_UTolima/blob/main/app_biomass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def agregar_capa_gee(ee_image, vis_params, nombre_capa):
    """Convierte una imagen de Earth Engine en una capa compatible con ipyleaflet."""
    map_id_dict = ee.Image(ee_image).getMapId(vis_params)
    tile_layer = L.TileLayer(
        url=map_id_dict['tile_fetcher'].url_format,
        attribution='Google Earth Engine',
        name=nombre_capa
    )
    return tile_layer

import ee
import os
import json
import ipyleaflet as L
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# --- INICIALIZACIÓN INTELIGENTE CON CUENTA DE SERVICIO PARA BINDER ---
try:
    # 1. Intentamos inicializar de forma regular (útil si estás corriendo localmente en Colab)
    ee.Initialize(project='ee-ygarces')
except Exception:
    try:
        # 2. Si falla (como pasará en MyBinder), buscamos la clave secreta en las variables del entorno
        # Binder inyectará automáticamente el secreto de GitHub aquí al compilar
        if 'EE_PRIVATE_KEY' in os.environ:
            private_key_dict = json.loads(os.environ['EE_PRIVATE_KEY'])

            # Extraemos los datos del JSON de la Cuenta de Servicio
            client_email = private_key_dict['client_email']

            # Creamos las credenciales firmadas para Earth Engine
            credentials = ee.ServiceAccountCredentials(client_email, key_data=os.environ['EE_PRIVATE_KEY'])

            # Inicializamos el servidor sin pedir interacción del usuario final
            ee.Initialize(credentials=credentials, project='ee-ygarces')
            print("🔑 Conexión exitosa en Binder mediante Cuenta de Servicio de Google Cloud.")
        else:
            # Fallback seguro por si ejecutas en un entorno limpio que requiera token web
            ee.Authenticate()
            ee.Initialize(project='ee-ygarces')
    except Exception as e:
        print(f"❌ Error crítico de autenticación en GEE: {e}")

def obtener_capa_gee(ee_image, vis_params, nombre_capa):
    map_id_dict = ee.Image(ee_image).getMapId(vis_params)
    return L.TileLayer(
        url=map_id_dict['tile_fetcher'].url_format,
        attribution='Google Earth Engine',
        name=nombre_capa
    )

# 1. ENTORNO GEOGRÁFICO
municipios_col = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                   .filter(ee.Filter.And(
                       ee.Filter.eq('ADM0_NAME', 'Colombia'),
                       ee.Filter.eq('ADM1_NAME', 'Tolima')
                   ))
municipios_geojson = municipios_col.geometry().getInfo()

# 2. MAPA BASE (Ajustamos la altura para que encaje estéticamente con el panel izquierdo)
mapa = L.Map(center=(4.0, -75.25), zoom=8, scroll_wheel_zoom=True, layout=widgets.Layout(height='480px', width='100%'))
mapa.add_layer(L.TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attribution='ESRI World Imagery',
    name='SATELLITE_BASE'
))
mapa.add_layer(L.GeoJSON(
    data=municipios_geojson,
    style={'color': '#ef4444', 'fillOpacity': 0.0, 'weight': 1.5},
    name='Municipios del Tolima'
))

# 3. INTERFAZ, CONTROLES Y PANEL INSTITUCIONAL IZQUIERDO
selector_anio = widgets.Dropdown(
    options=[str(anio) for anio in range(2017, 2026)],
    value='2022',
    description='Año:',
    layout=widgets.Layout(width='35%')
)
btn_limpiar_historial = widgets.Button(
    description='Limpiar Historial',
    button_style='warning',
    layout=widgets.Layout(width='40%')
)
consola_estado = widgets.Label(
    value='Estado: Esperando que dibujes un rectángulo...',
    layout=widgets.Layout(margin='5px 0 0 0')
)
zona_graficos = widgets.Output(layout=widgets.Layout(width='100%', margin='20px 0 0 0'))

control_dibujo = L.DrawControl(
    rectangle={'shapeOptions': {'color': '#3b82f6', 'fillOpacity': 0.1, 'weight': 2}},
    polygon={}, polyline={}, circle={}, circlemarker={}, marker={}
)
mapa.add_control(control_dibujo)
mapa.add_control(L.LayersControl(position='topright'))

# --- DISEÑO DEL PANEL INSTITUCIONAL IZQUIERDO (#A50000) ---
html_panel_izquierdo = widgets.HTML(
    value="""
    <div style='background-color: #A50000; color: white; padding: 25px; border-radius: 8px; height: 430px; font-family: sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
        <div style='text-align: center; margin-bottom: 15px;'>
            <img src='https://administrativos.ut.edu.co/images/Home/simbolos/logo_oficial.png' style='height: 110px; background-color: white; padding: 10px; border-radius: 5px;'/>
        </div>
        <h2 style='text-align: center; margin: 10px 0 5px 0; font-size: 20px; font-weight: bold; letter-spacing: 0.5px;'>Universidad del Tolima</h2>
        <p style='text-align: center; font-style: italic; font-size: 13px; opacity: 0.9; margin-bottom: 20px;'>"Construimos la Universidad que Soñamos"</p>
        <hr style='border: 0; border-top: 1px solid rgba(255,255,255,0.3); margin: 15px 0;'/>
        <div style='margin-top: 15px; font-size: 14px; line-height: 1.5;'>
            <p><strong>Investigador Principal:</strong><br>Yeison Alberto Garcés Gómez</p>
            <p><strong>Plataforma Analítica:</strong><br>Estimación Inteligente de Biomasa Aérea mediante Google AlphaEarth Satellite Embeddings y Algoritmos de Regresión No Lineal.</p>
            <p style='font-size: 12px; opacity: 0.8; margin-top: 25px;'>Sede Central Santa Helena, Ibagué, Tolima, Colombia.</p>
        </div>
    </div>
    """,
    layout=widgets.Layout(width='100%', padding='0px 10px 0px 0px')
)

capas_dinamicas = []
historial_biomasa_total = {}
ultima_geometria_roi = None

def resetear_historial(_):
    global historial_biomasa_total, ultima_geometria_roi
    historial_biomasa_total.clear()
    ultima_geometria_roi = None
    zona_graficos.clear_output()
    control_dibujo.clear()
    consola_estado.value = "🔄 Historial y mapa borrados. Dibuja un nuevo rectángulo."
btn_limpiar_historial.on_click(resetear_historial)

# 4. FUNCIÓN NÚCLEO
def procesar_analisis_espacial(roi, anio_str):
    global capas_dinamicas, historial_biomasa_total

    for capa in capas_dinamicas:
        if capa in mapa.layers: mapa.remove_layer(capa)
    capas_dinamicas.clear()
    zona_graficos.clear_output()

    anio_int = int(anio_str)
    es_prediccion_futura = anio_int >= 2023
    anio_entrenamiento = "2021" if es_prediccion_futura else anio_str

    if es_prediccion_futura:
        consola_estado.value = f"🔮 MODO PREDICCIÓN: Calibrando con 2021 para proyectar mapa {anio_str}..."
    else:
        consola_estado.value = f"⏳ Paso 1/3: Cargando datos históricos para el año {anio_str}..."

    region_sample = roi.bounds()
    fecha_inicio_target = f"{anio_str}-01-01"
    fecha_fin_target = f"{anio_str}-12-31"
    fecha_inicio_base = f"{anio_entrenamiento}-01-01"
    fecha_fin_base = f"{anio_entrenamiento}-12-31"

    try:
        # --- 4.1 PASO VISUAL: SENTINEL-2 Y MÁSCARA DE AGUA ---
        s2_composicion = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                           .filterBounds(roi).filterDate(fecha_inicio_target, fecha_fin_target) \
                           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)).median().clip(roi)

        capa_s2 = obtener_capa_gee(s2_composicion, {'bands': ['B4', 'B3', 'B2'], 'max': 2000, 'min': 0}, f'Sentinel-2 ({anio_str})')
        mapa.add_layer(capa_s2); capas_dinamicas.append(capa_s2)

        mascara_agua = s2_composicion.select('B12').lt(350)
        agua_negra_img = ee.Image.constant(0).visualize(palette=['#000000']).updateMask(mascara_agua)
        capa_agua_negra = obtener_capa_gee(agua_negra_img, {}, f'Cuerpos de Agua (Ref Negra)')
        mapa.add_layer(capa_agua_negra); capas_dinamicas.append(capa_agua_negra)

        # --- 4.2 PREPARACIÓN DE ENTRADAS ---
        dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation').clip(roi)
        slope = ee.Terrain.slope(dem).rename('slope')

        embeddings_target_col = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterBounds(roi).filterDate(fecha_inicio_target, fecha_fin_target)
        embedding_target_img = embeddings_target_col.median().clip(roi)
        band_names_embeddings = embedding_target_img.bandNames().getInfo()

        features_target_img = embedding_target_img.addBands(dem).addBands(slope)
        columnas_X_esperadas = band_names_embeddings + ['elevation', 'slope']

        capa_embed = obtener_capa_gee(embedding_target_img, {'bands': band_names_embeddings[:3], 'min': -1, 'max': 1}, f'Embeddings ({anio_str})')
        mapa.add_layer(capa_embed); capas_dinamicas.append(capa_embed)

        biomasa_train_img = ee.Image(f'ESA/CCI/Above_Ground_Biomass/V6_0/{anio_entrenamiento}').select('agb').clip(roi)

        # --- 4.3 CONSTRUCCIÓN DE LA CAPA TARGET Y CAPA BASE COMPARATIVA ---
        paleta_biomasa = ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#1a9850']

        if es_prediccion_futura:
            embeddings_base_col = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterBounds(roi).filterDate(fecha_inicio_base, fecha_fin_base)
            embedding_base_img = embeddings_base_col.median().clip(roi)
            image_for_sampling = biomasa_train_img.addBands(embedding_base_img).addBands(dem).addBands(slope)

            sample_points = image_for_sampling.sample(region=region_sample, scale=100, numPixels=3000, seed=42, geometries=True)
            features_json = sample_points.getInfo()['features']
            lista_puntos = [f['properties'] for f in features_json]
            df = pd.DataFrame(lista_puntos).dropna()
            columnas_X = [col for col in columnas_X_esperadas if col in df.columns]

            clasificador_gee = ee.Classifier.smileRandomForest(100).setOutputMode('REGRESSION').train(
                features=sample_points, classProperty='agb', inputProperties=columnas_X
            )
            biomasa_final_img = features_target_img.select(columnas_X).classify(clasificador_gee).rename('agb')

            info_p95_train = biomasa_train_img.reduceRegion(reducer=ee.Reducer.percentile([95]), geometry=roi, scale=100, maxPixels=1e9).getInfo()
            max_train = info_p95_train.get('agb_p95', 120) or 120

            capa_base_entrenamiento = obtener_capa_gee(
                biomasa_train_img,
                {'min': 0, 'max': max_train, 'palette': ['#f7f7f7', '#cccccc', '#969696', '#636363', '#252525']},
                f'🔍 Biomasa ESA Base ({anio_entrenamiento} - Gris)'
            )
            mapa.add_layer(capa_base_entrenamiento); capas_dinamicas.append(capa_base_entrenamiento)

            nombre_capa_biomasa = f'🔮 Biomasa PREDICHA ({anio_str})'
        else:
            image_for_sampling = biomasa_train_img.addBands(embedding_target_img).addBands(dem).addBands(slope)
            sample_points = image_for_sampling.sample(region=region_sample, scale=100, numPixels=3000, seed=42, geometries=True)
            features_json = sample_points.getInfo()['features']
            lista_puntos = [f['properties'] for f in features_json]
            df = pd.DataFrame(lista_puntos).dropna()
            columnas_X = [col for col in columnas_X_esperadas if col in df.columns]

            biomasa_final_img = biomasa_train_img
            nombre_capa_biomasa = f'Biomasa ESA Real ({anio_str})'

        info_p95 = biomasa_final_img.reduceRegion(reducer=ee.Reducer.percentile([95]), geometry=roi, scale=100, maxPixels=1e9).getInfo()
        max_grafico = info_p95.get('agb_p95', 120) or 120

        capa_biomasa = obtener_capa_gee(biomasa_final_img, {'min': 0, 'max': max_grafico, 'palette': paleta_biomasa}, nombre_capa_biomasa)
        mapa.add_layer(capa_biomasa); capas_dinamicas.append(capa_biomasa)

        # --- 4.4 REGRESIÓN DE VALIDACIÓN EN LOCAL Y MÉTRICAS ---
        X = df[columnas_X]
        y = df['agb']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        reg_model_topo = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        reg_model_topo.fit(X_train, y_train)
        y_pred = reg_model_topo.predict(X_test)
        r2_topo = r2_score(y_test, y_pred)

        imagen_biomasa_total = biomasa_final_img.multiply(ee.Image.pixelArea().divide(10000))
        suma_biomasa = imagen_biomasa_total.reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=100, maxPixels=1e9).getInfo()
        biomasa_total_calculada = suma_biomasa.get('agb', 0)
        historial_biomasa_total[anio_str] = biomasa_total_calculada

        # --- 4.5 CONFIGURACIÓN DE GRÁFICOS ---
        with zona_graficos:
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 5))

            # G1: Importancia de Variables
            importances = reg_model_topo.feature_importances_
            feat_importances = pd.Series(importances, index=columnas_X).sort_values(ascending=False)
            feat_importances.head(15).plot(kind='bar', color='teal', ax=ax1)
            ax1.set_title(f'Importancia de Variables (Base: {anio_entrenamiento})')
            ax1.set_ylabel('Importancia Relativa')
            ax1.grid(axis='y', linestyle='--', alpha=0.4)

            # G2: Ajuste de Regresión
            ax2.scatter(y_test, y_pred, alpha=0.6, color='darkgreen')
            ax2.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
            ax2.set_xlabel('Biomasa Real (Mg/ha)')
            ax2.set_ylabel('Biomasa Predicha (Mg/ha)')
            ax2.set_title(f'Validación del Modelo Base (R²: {r2_topo:.4f})')
            ax2.grid(True, linestyle='--', alpha=0.4)

            # G3: Serie Temporal
            anios_ordenados = sorted(historial_biomasa_total.keys(), key=int)
            valores_biomasa = [historial_biomasa_total[a] for a in anios_ordenados]
            colores_nodos = ['#c2410c' if int(a) < 2023 else '#7c3aed' for a in anios_ordenados]

            ax3.plot(anios_ordenados, valores_biomasa, color='#94a3b8', linestyle='-', linewidth=2, zorder=1)
            for a, v, col in zip(anios_ordenados, valores_biomasa, colores_nodos):
                etiqueta = f"🔮 {int(v):,}\nMg" if int(a) >= 2023 else f"{int(v):,}\nMg"
                ax3.scatter(a, v, color=col, s=100, zorder=2)
                ax3.annotate(etiqueta, xy=(a, v), xytext=(0, 10), textcoords='offset points', ha='center', fontsize=9, weight='bold', color=col)

            ax3.set_title('Historial y Proyecciones de Biomasa Total (Mg)')
            ax3.set_xlabel('Año de Análisis')
            ax3.set_ylabel('Biomasa Total Acumulada (Mg)')
            ax3.grid(True, linestyle='--', alpha=0.4)
            if valores_biomasa:
                ax3.set_ylim(min(valores_biomasa)*0.85, max(valores_biomasa)*1.15)

            plt.tight_layout()
            plt.show()
            plt.close(fig)

        if es_prediccion_futura:
            consola_estado.value = f"🔮 MAPA PROYECTADO ({anio_str}) | Total: {int(biomasa_total_calculada):,} Mg | R² Modelo Base (2021): {r2_topo:.4f}"
        else:
            consola_estado.value = f"🟢 Datos históricos de {anio_str} | Biomasa Total Real: {int(biomasa_total_calculada):,} Mg | R²: {r2_topo:.4f}"

    except Exception as e:
        consola_estado.value = f"❌ Error: {str(e)[:60]}"
        print("Error detallado:", e)

# 5. MANEJADORES DE EVENTOS
def al_dibujar_rectangulo(target, action, geo_json):
    global ultima_geometria_roi
    if action == 'created' and geo_json['geometry']['type'] == 'Polygon':
        coordenadas = geo_json['geometry']['coordinates'][0]
        ultima_geometria_roi = ee.Geometry.Polygon(coordenadas)
        procesar_analisis_espacial(ultima_geometria_roi, selector_anio.value)

control_dibujo.on_draw(al_dibujar_rectangulo)

def al_cambiar_anio(change):
    global ultima_geometria_roi
    if change['new']:
        nuevo_anio = change['new']
        if ultima_geometria_roi is not None:
            procesar_analisis_espacial(ultima_geometria_roi, nuevo_anio)
        else:
            consola_estado.value = f"Año cambiado a {nuevo_anio}. Dibuja un rectángulo para procesar."

selector_anio.observe(al_cambiar_anio, names='value')

# 6. CONSTRUCCIÓN DE LA ARQUITECTURA VISUAL (SPLIT 50/50)
# Controles compactos para la parte superior del mapa
barra_controles_mapa = widgets.HBox([selector_anio, btn_limpiar_historial], layout=widgets.Layout(justify_content='space-between', margin='0 0 5px 0'))
columna_derecha_mapa = widgets.VBox([barra_controles_mapa, mapa, consola_estado], layout=widgets.Layout(width='50%', padding='0px 0px 0px 10px'))
columna_izquierda_institucional = widgets.VBox([html_panel_izquierdo], layout=widgets.Layout(width='50%'))

# Contenedor principal que une los dos lados en paralelo
interfaz_split_principal = widgets.HBox([columna_izquierda_institucional, columna_derecha_mapa], layout=widgets.Layout(width='100%'))

# DESPLIEGUE FINAL DE LA APP COMPLETAMENTE CONFIGURADA
display(interfaz_split_principal, zona_graficos)


Output(layout=Layout(margin='20px 0 0 0', width='100%'))